# Присвоєння сегментів за правилами

Без KMeans і без sklearn-дерева: використовуються лише функції з `utils.py`
та збережені в `output/` правила і конфігурація з pipeline.

In [ ]:
import json
import importlib
from pathlib import Path

import pandas as pd
from IPython.display import display

import utils
importlib.reload(utils)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

In [ ]:
DATA_PATH = Path(r'C:\repo\data-science\Nazar Hnatyshyn\RETAIL\(DS-1052) Vehicle market analysis\df_for_clustering.xlsx')
SHEET_NAME = 0
OUTPUT_DIR = Path("output")

config = utils.load_json(OUTPUT_DIR / "segmentation_config.json")
tree_rules = utils.load_json(OUTPUT_DIR / "tree_rules.json")
print(json.dumps(config, ensure_ascii=False, indent=2))
print("Правил дерева:", len(tree_rules["rules"]))

In [ ]:
raw = utils.load_raw(DATA_PATH, SHEET_NAME)
print(f"Рядків (ТЗ): {len(raw):,} | клієнтів: {raw['user_id'].nunique():,}")

vehicles = utils.prepare_vehicle_data(raw)
customers = utils.aggregate_customers(vehicles, age_median=config["age_median"])
print("Клієнтів після агрегації:", f"{len(customers):,}")
display(customers.head())

In [ ]:
scored = utils.assign_final_segments(
    customers, tree_rules, ev_rule=config["ev_rule"], business_threshold=config["business_threshold"]
)
sizes = scored["FINAL_SEGMENT"].value_counts().rename_axis("FINAL_SEGMENT").reset_index(name="CLIENTS")
sizes["PCT"] = (sizes["CLIENTS"] / len(scored) * 100).round(2)
print(f"Сума по сегментах: {sizes['CLIENTS'].sum():,}")
display(sizes)

In [ ]:
scored[utils.output_columns(scored)].to_csv(
    OUTPUT_DIR / "segments_scored.csv.gz", index=False, compression="gzip", encoding="utf-8-sig"
)
print("Збережено:", OUTPUT_DIR / "segments_scored.csv.gz")

## Перевірка: score = pipeline

In [ ]:
reference = pd.read_csv(OUTPUT_DIR / "customer_segments.csv.gz", usecols=["user_id", "FINAL_SEGMENT"])
current = pd.read_csv(OUTPUT_DIR / "segments_scored.csv.gz", usecols=["user_id", "FINAL_SEGMENT"])

cmp = current.merge(reference, on="user_id", how="outer", suffixes=("_SCORE", "_PIPELINE"), indicator=True)
match = cmp["_merge"].eq("both") & cmp["FINAL_SEGMENT_SCORE"].eq(cmp["FINAL_SEGMENT_PIPELINE"])
print(f"Збіг: {match.sum():,} / {len(cmp):,} = {match.mean():.4%}")

if not match.all():
    print("Розбіжності:")
    display(cmp.loc[~match].head(50))
assert match.all(), "Score не збігається з pipeline"
print("OK: score повністю відтворює pipeline")